In [44]:
import numpy as np
import pandas as pd
from sklearn.utils import resample
import transformers
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from peft import LoraConfig, get_peft_model, TaskType
from pyfaidx import Fasta
import pandas as pd
import torch
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, confusion_matrix, f1_score, classification_report, average_precision_score
from sklearn.linear_model import LogisticRegression
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

In [ ]:
%pip install xgboost

In [ ]:
genome = Fasta(r"Homo_sapiens_CFTR_sequence.fa")

In [ ]:
#genome = Fasta(r"Homo_sapiens_CFTR_sequence.fa")

df = pd.read_csv(r"cleaned_datasets/final_cftr_dataset.csv")

df_balanced = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df_major = df[df["cause"] == 1]
df_minor = df[df["cause"] == 0]

df_minor_upsampled = resample(df_minor,
                             replace=True,
                             n_samples=len(df_major),
                             random_state=42)

df_balanced = pd.concat([df_major, df_minor_upsampled])

In [4]:
df_balanced.head()

,chr,pos,ref,alt,cause
0,NC_000007.14,117535377,C,T,1
1,NC_000007.14,117642579,G,T,1
2,NC_000007.14,117665565,G,T,1
3,NC_000007.14,117587834,G,A,1
4,NC_000007.14,117504314,C,T,1


In [5]:
df_model = df_balanced[["chr", "pos", "ref", "alt", "cause"]]
df_model.rename(columns={"cause": "label"}, inplace=True)

In [6]:
df_model[df_model["label"].isna()]

,chr,pos,ref,alt,label


In [ ]:
df_model = df_model.dropna(subset=["chr", "pos", "ref", "alt"])

In [7]:
df_model.head()

,chr,pos,ref,alt,label
0,NC_000007.14,117535377,C,T,1
1,NC_000007.14,117642579,G,T,1
2,NC_000007.14,117665565,G,T,1
3,NC_000007.14,117587834,G,A,1
4,NC_000007.14,117504314,C,T,1


In [8]:
df_model["ref"] = df_model["ref"].astype(str).str.upper()
df_model["alt"] = df_model["alt"].astype(str).str.upper()
df_model["pos"] = df_model["pos"].astype(int)

In [9]:
df_model = df_model[
    df_model["ref"].str.fullmatch(r"[ACGT]+", na=False) &
    df_model["alt"].str.fullmatch(r"[ACGT]+", na=False)
].reset_index(drop=True)

In [10]:
df_model["variant_id"] = (
    df["chr"] + "_" +
    df["pos"].astype(str) + "_" +
    df["ref"] + "_" +
    df["alt"]
)

df_model["variant_id"].value_counts()

variant_id
NC_000007.14_117509228_T_G                                                                                        1
NC_000007.14_117531121_C_T                                                                                        1
NC_000007.14_117627546_A_T                                                                                        1
NC_000007.14_117542016_G_GA                                                                                       1
NC_000007.14_117591983_AAAATGGAACATTTAAAGAAAGCTGACAAAATATTAATTTTGCATGAAGGTAGCAGCTATTTTTATGGGACATTTTCAGAACTCC_A    1
                                                                                                                 ..
NC_000007.14_117548733_A_G                                                                                        1
NC_000007.14_117611555_A_G                                                                                        1
NC_000007.14_117509109_AT_A                                  

In [11]:
df_model.shape

(2301, 6)

In [12]:
def extract_window(seq, pos, ref_len, window=50):
    start = max(0, pos - window)
    end = min(len(seq), pos + ref_len + window)
    real_window =  seq[start:end]
    rel_pos = pos - start
    return real_window, rel_pos

In [13]:
def apply_mutation(seq, ref, alt, pos):
    if not isinstance(ref, str) or not isinstance(alt, str):
        return None
    return seq[:pos] + alt + seq[pos+len(ref):]

In [14]:
def create_mut_seq(row, genome, window=50):
    try:
        chrom = row["chr"]
        pos = row["pos"] - 1
        ref = row["ref"]
        alt = row["alt"]

        full_seq = genome  # convert pyfaidx → string
        
        if pos < 0 or pos + len(ref) > len(full_seq):
            return None, None, False

        
        genome_ref = full_seq[pos : pos + len(ref)]
        if genome_ref != ref:
            return None, None, False
        
        
        ref_seq, rel_pos = extract_window(full_seq, pos, len(ref), window)

        mut_seq = apply_mutation(ref_seq, ref, alt, rel_pos)

        if ref_seq[rel_pos:rel_pos+len(ref)] != ref:
            return None, None, False

        if mut_seq[rel_pos:rel_pos+len(alt)] != alt:
            return None, None, False

        return ref_seq, mut_seq, True

    except Exception as e:
        print("Error:", e)
        return None, None, False

In [15]:
chr_seq = str(genome["NC_000007.14"])

results = df_model.apply(
    lambda row: pd.Series(
        create_mut_seq(row, chr_seq, window=50),
        index=["ref_seq", "mut_seq", "match"]
    ),
    axis=1
)

df_model = pd.concat([df_model, results], axis=1)

print("Before filter:", df_model.shape)
print(df_model["match"].value_counts(dropna=False))

# df_model = df_model[df_model["match"] == True].reset_index(drop=True)

print("After filter:", df_model.shape)

Before filter: (2301, 9)
match
True     1972
False     329
Name: count, dtype: int64
After filter: (2301, 9)


In [16]:
df_model = df_model[df_model["match"] == True].reset_index(drop=True)

In [17]:
df_model.reset_index(drop=True, inplace=True)

In [19]:
df_model.columns

Index(['chr', 'pos', 'ref', 'alt', 'label', 'variant_id', 'ref_seq', 'mut_seq',
       'match'],
      dtype='object')

In [20]:
df_model.to_csv("cftr_clean_final_mutated.csv", index=False)

In [18]:
def reverse_complement(seq):
    comp = {'A':'T','T':'A','C':'G','G':'C'}
    return ''.join(comp.get(b, 'N') for b in reversed(seq))

df_model["rc_sequence"] = df_model["sequence"].apply(reverse_complement)

In [ ]:
#df_model.rename(columns={"cause": "label"}, inplace=True)

df_final = pd.concat([
    df_model[["sequence", "label"]],
    df_model[["rc_sequence", "label"]].rename(columns={"rc_sequence": "sequence"})
], ignore_index=True)

In [29]:
# Group nearby variants together so overlapping windows do not leak into both train/test
BIN_SIZE = 1000
df_model["bin"] = (df_model["pos"] // BIN_SIZE).astype(int)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_model, y=df_model["label"], groups=df_model["bin"]))

train_df = df_model.iloc[train_idx].reset_index(drop=True)
test_df = df_model.iloc[test_idx].reset_index(drop=True)

print("Train:", train_df.shape, train_df["label"].value_counts().to_dict())
print("Test :", test_df.shape, test_df["label"].value_counts().to_dict())

# No exact variant leakage
#assert set(train_df["variant_id"]).isdisjoint(set(test_df["variant_id"]))

Train: (1572, 10) {1: 1044, 0: 528}
Test : (400, 10) {1: 260, 0: 140}


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "zhihan1996/DNABERT-2-117M"    
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(model_name, trust_remote_code=True).to(device)
model.eval()

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def embed_sequences(sequences, batch_size=16, max_length=512):
    all_emb = []

    for i in range(0, len(sequences), batch_size):
        batch = list(sequences[i:i + batch_size])

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # safer than outputs[0]
        last_hidden = outputs.last_hidden_state
        pooled = mean_pool(last_hidden, inputs["attention_mask"])
        all_emb.append(pooled.cpu().numpy())

    return np.vstack(all_emb)

c:\Users\admin\anaconda3\envs\dnabert2_cftr\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Explicitly passing a `revision` is encouraged when loading a configuration with custom code to ensure no malicious code has been contributed in a newer revision.
Explicitly passing a `revision` is encouraged when loading a model with custom code to ensure no malicious code has been contributed in a newer revision.
C:\Users\admin/.cache\huggingface\modules\transformers_modules\zhihan1996\DNABERT-2-117M\7bce263b15377fc15361f52cfab88f8b586abda0\bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of the model checkpoint at zhihan1996/DNABERT-2-11

In [57]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "zhihan1996/DNABERT-2-117M"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(model_name, trust_remote_code=True).to(device)
model.eval()

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def embed_sequences(sequences, batch_size=16, max_length=512):
    all_emb = []

    for i in range(0, len(sequences), batch_size):
        batch = list(sequences[i:i + batch_size])

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # safer than outputs[0]
        last_hidden = outputs.last_hidden_state
        pooled = mean_pool(last_hidden, inputs["attention_mask"])
        all_emb.append(pooled.cpu().numpy())

    return np.vstack(all_emb)

Explicitly passing a `revision` is encouraged when loading a configuration with custom code to ensure no malicious code has been contributed in a newer revision.
Explicitly passing a `revision` is encouraged when loading a model with custom code to ensure no malicious code has been contributed in a newer revision.
C:\Users\admin/.cache\huggingface\modules\transformers_modules\zhihan1996\DNABERT-2-117M\7bce263b15377fc15361f52cfab88f8b586abda0\bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of the model checkpoint at zhihan1996/DNABERT-2-117M were not used when initializing BertModel: ['cls.predictions.decoder.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias']
- This IS expected i

In [32]:
train_df


,chr,pos,ref,alt,label,variant_id,ref_seq,mut_seq,match,bin
0,NC_000007.14,117535377,C,T,1,NC_000007.14_117531121_C_T,AGGCGTCTGCCTTCTGTGGACTTGGTTTCCTGATAGTCCTTGCCCT...,AGGCGTCTGCCTTCTGTGGACTTGGTTTCCTGATAGTCCTTGCCCT...,True,117535
1,NC_000007.14,117642579,G,T,1,NC_000007.14_117627546_A_T,ATGGTGTGTCTTGGGATTCAATAACTTTGCAACAGTGGAGGAAAGC...,ATGGTGTGTCTTGGGATTCAATAACTTTGCAACAGTGGAGGAAAGC...,True,117642
2,NC_000007.14,117665565,G,T,1,NC_000007.14_117542016_G_GA,TTCTCTGTGAACACAGGATAGAAGCAATGCTGGAATGCCAACAATT...,TTCTCTGTGAACACAGGATAGAAGCAATGCTGGAATGCCAACAATT...,True,117665
3,NC_000007.14,117587834,G,A,1,NC_000007.14_117591983_AAAATGGAACATTTAAAGAAAGC...,GGTGGAATCACACTGAGTGGAGGTCAACGAGCAAGAATTTCTTTAG...,GGTGGAATCACACTGAGTGGAGGTCAACGAGCAAGAATTTCTTTAG...,True,117587
4,NC_000007.14,117559466,T,A,0,NC_000007.14_117548650_G_T,CTGAGCGTGATTTGATAATGACCTAATAATGATGGGTTTTATTTCC...,CTGAGCGTGATTTGATAATGACCTAATAATGATGGGTTTTATTTCC...,True,117559
...,...,...,...,...,...,...,...,...,...,...
1567,NC_000007.14,117592114,TTT,TT,1,NC_000007.14_117540236_ATATTCACCACCAT_ATATTCAC...,CTCCAAAATCTACAGCCAGACTTTAGCTCAAAACTCATGGGATGTG...,CTCCAAAATCTACAGCCAGACTTTAGCTCAAAACTCATGGGATGTG...,True,117592
1568,NC_000007.14,117603689,CA,C,1,NC_000007.14_117614686_A_G,TAGCCGACACTTTGCTTGCTATGGGATTCTTCAGAGGTCTACCACT...,TAGCCGACACTTTGCTTGCTATGGGATTCTTCAGAGGTCTACCACT...,True,117603
1569,NC_000007.14,117652886,C,G,0,NC_000007.14_117536626_A_G,ctatagaaagtatttattttttctggaacATTTAGAAAAAACTTGG...,ctatagaaagtatttattttttctggaacATTTAGAAAAAACTTGG...,True,117652
1570,NC_000007.14,117603611,T,TG,1,NC_000007.14_117540199_TC_T,CTCATAGTAGAAATAACAGCTATGCAGTGATTATCACCAGCACCAG...,CTCATAGTAGAAATAACAGCTATGCAGTGATTATCACCAGCACCAG...,True,117603


In [34]:
def embed_sequences(sequences, batch_size=16, max_length=512):
    all_emb = []

    for i in range(0, len(sequences), batch_size):
        batch = sequences[i:i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # ✅ FIX HERE
        last_hidden = outputs[0]

        # mean pooling
        attention_mask = inputs["attention_mask"]
        mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()

        summed = (last_hidden * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)

        pooled = summed / counts

        all_emb.append(pooled.cpu().numpy())

    return np.vstack(all_emb)

In [36]:
X_train_ref = embed_sequences(train_df["ref_seq"].tolist())
X_train_mut = embed_sequences(train_df["mut_seq"].tolist())

In [37]:
X_train = np.hstack([
    X_train_ref,
    X_train_mut,
    X_train_mut - X_train_ref,
    np.abs(X_train_mut - X_train_ref)
])

In [40]:
X_test_ref = embed_sequences(test_df["ref_seq"].tolist())
X_test_mut = embed_sequences(test_df["mut_seq"].tolist())

X_test = np.hstack([
    X_test_ref,
    X_test_mut,
    X_test_mut - X_test_ref,
    np.abs(X_test_mut - X_test_ref)
])

In [42]:
y_train = train_df["label"].values
y_test = test_df["label"].values

In [43]:
print(X_train.shape, X_test.shape)

(1572, 3072) (400, 3072)


In [58]:
lr = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    n_jobs=-1
)
lr.fit(X_train, y_train)

proba_lr = lr.predict_proba(X_test)[:, 1]
pred_lr = (proba_lr >= 0.5).astype(int)

print("Logistic Regression")
print(classification_report(y_test, pred_lr, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, proba_lr))
print("PR-AUC :", average_precision_score(y_test, proba_lr))
print(confusion_matrix(y_test, pred_lr))

Logistic Regression
              precision    recall  f1-score   support

           0     0.3222    0.6214    0.4244       140
           1     0.5923    0.2962    0.3949       260

    accuracy                         0.4100       400
   macro avg     0.4573    0.4588    0.4096       400
weighted avg     0.4978    0.4100    0.4052       400

ROC-AUC: 0.4213736263736264
PR-AUC : 0.6021893762229387
[[ 87  53]
 [183  77]]


In [46]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    tree_method="hist",
    random_state=42
)

xgb.fit(X_train, y_train)

proba_xgb = xgb.predict_proba(X_test)[:, 1]
pred_xgb = (proba_xgb >= 0.5).astype(int)

print("XGBoost")
print(classification_report(y_test, pred_xgb, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, proba_xgb))
print("PR-AUC :", average_precision_score(y_test, proba_xgb))
print(confusion_matrix(y_test, pred_xgb))

XGBoost
              precision    recall  f1-score   support

           0     0.3889    0.1500    0.2165       140
           1     0.6561    0.8731    0.7492       260

    accuracy                         0.6200       400
   macro avg     0.5225    0.5115    0.4828       400
weighted avg     0.5626    0.6200    0.5627       400

ROC-AUC: 0.4932142857142857
PR-AUC : 0.6513276286576385
[[ 21 119]
 [ 33 227]]


In [47]:
import numpy as np

diff_norms = np.linalg.norm(X_train_mut - X_train_ref, axis=1)

print("Mean diff:", diff_norms.mean())
print("Std diff:", diff_norms.std())

Mean diff: 0.52844965
Std diff: 0.4689414


In [48]:
def build_pair_input(ref_seq, mut_seq):
    return ref_seq + " [SEP] " + mut_seq

In [49]:
train_df["pair_seq"] = train_df.apply(
    lambda row: build_pair_input(row["ref_seq"], row["mut_seq"]),
    axis=1
)

test_df["pair_seq"] = test_df.apply(
    lambda row: build_pair_input(row["ref_seq"], row["mut_seq"]),
    axis=1
)

In [50]:
X_train = embed_sequences(train_df["pair_seq"].tolist())
X_test  = embed_sequences(test_df["pair_seq"].tolist())

In [51]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000, class_weight="balanced")
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC :", average_precision_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.32      0.62      0.42       140
           1       0.59      0.30      0.39       260

    accuracy                           0.41       400
   macro avg       0.46      0.46      0.41       400
weighted avg       0.50      0.41      0.41       400

ROC-AUC: 0.4213736263736264
PR-AUC : 0.6021893762229387


In [52]:
WINDOW = 25

results = df_model.apply(
    lambda row: pd.Series(
        create_mut_seq(row, chr_seq, window=WINDOW),
        index=["ref_seq", "mut_seq", "match"]
    ),
    axis=1
)

# remove old columns first (IMPORTANT)
df_model = df_model.drop(columns=["ref_seq", "mut_seq", "match"], errors="ignore")

df_model = pd.concat([df_model, results], axis=1)

print("Before filter:", df_model.shape)
print(df_model["match"].value_counts())

df_model = df_model[df_model["match"] == True].reset_index(drop=True)

print("After filter:", df_model.shape)

Before filter: (1972, 10)
match
True    1972
Name: count, dtype: int64
After filter: (1972, 10)


In [53]:
from sklearn.model_selection import GroupShuffleSplit

BIN_SIZE = 1000
df_model["bin"] = (df_model["pos"] // BIN_SIZE).astype(int)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_model, groups=df_model["bin"]))

train_df = df_model.iloc[train_idx].reset_index(drop=True)
test_df  = df_model.iloc[test_idx].reset_index(drop=True)

In [54]:
def build_pair_input(ref_seq, mut_seq):
    return ref_seq + " [SEP] " + mut_seq

train_df["pair_seq"] = train_df.apply(
    lambda row: build_pair_input(row["ref_seq"], row["mut_seq"]),
    axis=1
)

test_df["pair_seq"] = test_df.apply(
    lambda row: build_pair_input(row["ref_seq"], row["mut_seq"]),
    axis=1
)

In [67]:
def embed_sequences(sequences, batch_size=16, max_length=512):
    all_emb = []

    for i in range(0, len(sequences), batch_size):
        batch = sequences[i:i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # ✅ FIX HERE
        last_hidden = outputs[0]

        attention_mask = inputs["attention_mask"]
        mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()

        summed = (last_hidden * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)

        pooled = summed / counts

        all_emb.append(pooled.cpu().numpy())

    return np.vstack(all_emb)

In [69]:
X_train = embed_sequences(train_df["pair_seq"].tolist())
X_test  = embed_sequences(test_df["pair_seq"].tolist())

y_train = train_df["label"].values
y_test  = test_df["label"].values

In [70]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=2000, class_weight="balanced")
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
y_proba = lr.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC :", average_precision_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.36      0.61      0.46       140
           1       0.67      0.43      0.52       260

    accuracy                           0.49       400
   macro avg       0.52      0.52      0.49       400
weighted avg       0.56      0.49      0.50       400

ROC-AUC: 0.5085989010989012
PR-AUC : 0.6642255633374403


In [71]:
def highlight_mutation(ref_seq, alt_seq, rel_pos, ref, alt):
    left = ref_seq[:rel_pos]
    right = ref_seq[rel_pos + len(ref):]

    # mutation token
    mut_token = f"[{ref}>{alt}]"

    return left + mut_token + right

In [72]:
def create_mut_seq_highlight(row, chr_seq, window=25):
    try:
        pos = int(row["pos"]) - 1
        ref = str(row["ref"]).upper()
        alt = str(row["alt"]).upper()

        full_seq = chr_seq

        if pos < 0 or pos + len(ref) > len(full_seq):
            return None, None, False

        if full_seq[pos:pos+len(ref)] != ref:
            return None, None, False

        # extract window
        ref_seq, rel_pos = extract_window(full_seq, pos, len(ref), window)

        # normal mutation sequence
        mut_seq = apply_mutation(ref_seq, ref, alt, rel_pos)

        # 🔥 NEW: highlighted sequence
        highlight_seq = highlight_mutation(ref_seq, mut_seq, rel_pos, ref, alt)

        return highlight_seq, True

    except Exception as e:
        print("Error:", e)
        return None, False

In [81]:
results = df_model.apply(
    lambda row: pd.Series(
        create_mut_seq_highlight(row, chr_seq, window=15),
        index=["highlight_seq", "match"]
    ),
    axis=1
)

df_model = df_model.drop(columns=["highlight_seq", "match"], errors="ignore")
df_model = pd.concat([df_model, results], axis=1)

df_model = df_model[df_model["match"] == True].reset_index(drop=True)

In [82]:
from sklearn.model_selection import GroupShuffleSplit

BIN_SIZE = 1000
df_model["bin"] = (df_model["pos"] // BIN_SIZE).astype(int)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_model, groups=df_model["bin"]))

train_df = df_model.iloc[train_idx].reset_index(drop=True)
test_df  = df_model.iloc[test_idx].reset_index(drop=True)

In [ ]:
df_model.to_csv("cleaned_datasets/cftr_clean_final_mutated.csv", index=False)

In [86]:
lr = LogisticRegression(max_iter=2000, class_weight="balanced")
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
y_proba = lr.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC :", average_precision_score(y_test, y_proba))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.52      0.62      0.57       140
           1       0.77      0.69      0.73       260

    accuracy                           0.67       400
   macro avg       0.65      0.66      0.65       400
weighted avg       0.68      0.67      0.67       400

ROC-AUC: 0.685989010989011
PR-AUC : 0.7800896941441401
[[ 87  53]
 [ 80 180]]


In [87]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    tree_method="hist",
    random_state=42
)

xgb.fit(X_train, y_train)

proba_xgb = xgb.predict_proba(X_test)[:, 1]
pred_xgb = (proba_xgb >= 0.5).astype(int)

print("XGBoost")
print(classification_report(y_test, pred_xgb, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, proba_xgb))
print("PR-AUC :", average_precision_score(y_test, proba_xgb))
print(confusion_matrix(y_test, pred_xgb))

XGBoost
              precision    recall  f1-score   support

           0     0.5522    0.2643    0.3575       140
           1     0.6907    0.8846    0.7757       260

    accuracy                         0.6675       400
   macro avg     0.6215    0.5745    0.5666       400
weighted avg     0.6422    0.6675    0.6293       400

ROC-AUC: 0.6762362637362637
PR-AUC : 0.7923348444585878
[[ 37 103]
 [ 30 230]]


In [88]:
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(len(y_train[y_train==0]) / len(y_train[y_train==1])),
    eval_metric="logloss",
    tree_method="hist",
    random_state=42
)

xgb.fit(X_train, y_train)

proba_xgb = xgb.predict_proba(X_test)[:, 1]
pred_xgb = (proba_xgb >= 0.5).astype(int)

print("XGBoost")
print(classification_report(y_test, pred_xgb, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, proba_xgb))
print("PR-AUC :", average_precision_score(y_test, proba_xgb))
print(confusion_matrix(y_test, pred_xgb))

XGBoost
              precision    recall  f1-score   support

           0     0.5729    0.3929    0.4661       140
           1     0.7204    0.8423    0.7766       260

    accuracy                         0.6850       400
   macro avg     0.6467    0.6176    0.6213       400
weighted avg     0.6688    0.6850    0.6679       400

ROC-AUC: 0.698434065934066
PR-AUC : 0.8063268117300842
[[ 55  85]
 [ 41 219]]


In [89]:
import numpy as np
from sklearn.metrics import f1_score

thresholds = np.linspace(0.1, 0.9, 50)

best_t = 0.5
best_f1 = 0

for t in thresholds:
    pred = (proba_xgb >= t).astype(int)
    f1 = f1_score(y_test, pred)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best threshold:", best_t)
print("Best F1:", best_f1)

Best threshold: 0.1
Best F1: 0.7890743550834598


In [93]:
threshold_xgb = 0.3

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(len(y_train[y_train==0]) / len(y_train[y_train==1])),
    eval_metric="logloss",
    tree_method="hist",
    random_state=42
)

xgb.fit(X_train, y_train)

proba_xgb = xgb.predict_proba(X_test)[:, 1]
pred_xgb = (proba_xgb >= threshold_xgb).astype(int)

print("XGBoost")
print(classification_report(y_test, pred_xgb, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, proba_xgb))
print("PR-AUC :", average_precision_score(y_test, proba_xgb))
print(confusion_matrix(y_test, pred_xgb))

XGBoost
              precision    recall  f1-score   support

           0     0.5652    0.0929    0.1595       140
           1     0.6631    0.9615    0.7849       260

    accuracy                         0.6575       400
   macro avg     0.6142    0.5272    0.4722       400
weighted avg     0.6289    0.6575    0.5660       400

ROC-AUC: 0.698434065934066
PR-AUC : 0.8063268117300842
[[ 13 127]
 [ 10 250]]


In [100]:

demo_df = df_model.sample(5, random_state=42)

for i, row in demo_df.iterrows():
    
    chrom = row["chr"]
    pos = int(row["pos"])
    ref = row["ref"]
    alt = row["alt"]
    label = row["label"]   # ground truth

    # --- STEP 1: build highlighted sequence ---
    ref_seq, rel_pos = extract_window(chr_seq, pos-1, len(ref), window=15)
    highlight_seq = highlight_mutation(ref_seq, ref_seq, rel_pos, ref, alt)

    # --- STEP 2: embedding ---
    X_demo = embed_sequences([highlight_seq])

    # --- STEP 3: prediction ---
    proba = xgb.predict_proba(X_demo)[0, 1]
    pred = 1 if proba >= 0.3 else 0   # using your chosen threshold

    # --- STEP 4: print nicely ---
    print(f"Variant {i+1}")
    print(f"Location : {chrom}:{pos}")
    print(f"Mutation : {ref} → {alt}")
    print(f"Sequence : {highlight_seq}")

    print(f"Prediction : {'Pathogenic' if pred==1 else 'Benign'}")
    print(f"Confidence : {proba:.3f}")
    print(f"Actual     : {'Pathogenic' if label==1 else 'Benign'}")
    
    if pred == label:
        print("Result     : ✅ Correct")
    else:
        print("Result     : ❌ Incorrect")

    print("-"*80)

Variant 753
Location : NC_000007.14:117559645
Mutation : AACTAGAAGAGGTAAGAAACTA → AACTA
Sequence : TCATCAAAGCATGCC[AACTAGAAGAGGTAAGAAACTA>AACTA]TGTGAAAACTTTTTG
Prediction : Pathogenic
Confidence : 0.985
Actual     : Pathogenic
Result     : ✅ Correct
--------------------------------------------------------------------------------
Variant 766
Location : NC_000007.14:117652890
Mutation : G → T
Sequence : AACTTGGATCCCTAT[G>T]AACAGTGGAGTGATC
Prediction : Pathogenic
Confidence : 0.943
Actual     : Pathogenic
Result     : ✅ Correct
--------------------------------------------------------------------------------
Variant 1657
Location : NC_000007.14:117530950
Mutation : TAT → G
Sequence : AGAATCATAGCTTCC[TAT>G]GACCCGGATAACAAG
Prediction : Pathogenic
Confidence : 0.952
Actual     : Pathogenic
Result     : ✅ Correct
--------------------------------------------------------------------------------
Variant 1289
Location : NC_000007.14:117559633
Mutation : T → G
Sequence : GATACAGAAGCGTCA[T>G]CAAAGCA

In [95]:
df_model.head()

,chr,pos,ref,alt,label,variant_id,bin,ref_seq,mut_seq,highlight_seq,match
0,NC_000007.14,117535377,C,T,1,NC_000007.14_117531121_C_T,117535,TTTCCTGATAGTCCTTGCCCTTTTTCAGGCTGGGCTAGGGAGAATG...,TTTCCTGATAGTCCTTGCCCTTTTTTAGGCTGGGCTAGGGAGAATG...,GTCCTTGCCCTTTTT[C>T]AGGCTGGGCTAGGGA,True
1,NC_000007.14,117642579,G,T,1,NC_000007.14_117627546_A_T,117642,TTTGCAACAGTGGAGGAAAGCCTTTGGAGTGATACCACAGGTGAGC...,TTTGCAACAGTGGAGGAAAGCCTTTTGAGTGATACCACAGGTGAGC...,TGGAGGAAAGCCTTT[G>T]GAGTGATACCACAGG,True
2,NC_000007.14,117665565,G,T,1,NC_000007.14_117542016_G_GA,117665,AATGCTGGAATGCCAACAATTTTTGGTGAGTCTTTATAACTTTACT...,AATGCTGGAATGCCAACAATTTTTGTTGAGTCTTTATAACTTTACT...,TGCCAACAATTTTTG[G>T]TGAGTCTTTATAACT,True
3,NC_000007.14,117587834,G,A,1,NC_000007.14_117591983_AAAATGGAACATTTAAAGAAAGC...,117587,AACGAGCAAGAATTTCTTTAGCAAGGTGAATAACTAATTATTGGTC...,AACGAGCAAGAATTTCTTTAGCAAGATGAATAACTAATTATTGGTC...,AATTTCTTTAGCAAG[G>A]TGAATAACTAATTAT,True
4,NC_000007.14,117504314,C,T,1,NC_000007.14_117480117_A_AG,117504,GCGCCTGGAATTGTCAGACATATACCAAATCCCTTCTGTTGATTCT...,GCGCCTGGAATTGTCAGACATATACTAAATCCCTTCTGTTGATTCT...,TTGTCAGACATATAC[C>T]AAATCCCTTCTGTTG,True


In [96]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)

X = embed_sequences(df_model["highlight_seq"].tolist())
y = df_model["label"].values
groups = df_model["bin"].values

from sklearn.metrics import roc_auc_score

scores = []

for train_idx, test_idx in gkf.split(X, y, groups):
    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(X_tr, y_tr)

    proba = clf.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, proba)
    
    scores.append(auc)

print("Mean ROC-AUC:", np.mean(scores))
print("Std:", np.std(scores))

Mean ROC-AUC: 0.6610555736410288
Std: 0.018922622524072664


In [91]:
df_final.sample(10)

NameError: name 'df_final' is not defined

In [ ]:
X = df_final["sequence"].values
y = df_final["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
model_name = "zhihan1996/DNABERT-2-117M"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True
)

model.eval()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
model.to(device)


In [ ]:
def get_embedding_batch(sequences, batch_size=32):
    embeddings = []

    for i in range(0, len(sequences), batch_size):
        batch = sequences[i:i+batch_size]

        inputs = tokenizer(
            list(batch),
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        cls_embeddings = outputs[0][:, 0, :]

        embeddings.append(cls_embeddings.cpu().numpy())

    return np.vstack(embeddings)

In [ ]:
X_train_emb = get_embedding_batch(X_train)
X_test_emb  = get_embedding_batch(X_test)

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_emb, y_train)

y_pred_lr = lr.predict(X_test_emb)

In [ ]:
print(torch.cuda.is_available())

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
)

In [ ]:
xgb.fit(X_train_emb, y_train)

y_pred_xgb = xgb.predict(X_test_emb)

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

print("XGBoost Results")
print(classification_report(y_test, y_pred_xgb))

print("ROC-AUC:",
      roc_auc_score(y_test, xgb.predict_proba(X_test_emb)[:,1]))

In [ ]:
df_model = df_balanced[["chr", "pos", "ref", "alt", "cause"]].copy()
df_model.rename(columns={"cause": "label"}, inplace=True)

In [ ]:
df_model.shape

In [ ]:
df_model = df_model[
    (df_model["ref"] != "nan") &
    (df_model["alt"] != "nan") &
    (df_model["ref"].str.len() > 0) &
    (df_model["alt"].str.len() > 0)
]

In [ ]:
df_model.shape

In [ ]:
def extract_window(seq, pos, window=50):
    start = max(0, pos - window)
    end = min(len(seq), pos + window)
    return seq[start:end], start, end

In [ ]:
def apply_mutation(seq, ref, alt, pos):
    if pos < 0 or pos + len(ref) > len(seq):
        return None

    if seq[pos:pos+len(ref)] != ref:
        return None

    return seq[:pos] + alt + seq[pos+len(ref):]

In [ ]:
OFFSET = 117287120

def create_ref_mut_seq(row, genome):
    try:
        chrom = row["chr"].replace("chr", "")
        pos = row["pos"] - OFFSET - 1
        ref = row["ref"]
        alt = row["alt"]
        
        if ref == "nan" or alt == "nan":
            return None, None

        full_seq = str(genome[chrom])
        
        if pos < 0 or pos >= len(full_seq):
            return None

        ref_seq, window_start, window_end = extract_window(full_seq, pos)
        rel_pos = pos - window_start

        mut_seq = apply_mutation(ref_seq, ref, alt, rel_pos)

        return ref_seq, mut_seq

    except Exception as e:
        print(f"Exception as {row}")
        return None

In [ ]:
df_model[["ref_mut", "alt_mut"]] = df_model.apply(
    lambda x: pd.Series(create_ref_mut_seq(x, genome)), axis = 1
)

In [ ]:
df_model = df_model.dropna(subset=["ref_mut", "alt_mut"]).reset_index(drop=True)

In [ ]:
df_model.shape